# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [5]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [4]:
import torch

# 线索 3：数据张量 (保持这些数据不变)
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

# 1. 创建三个待学习参数
# 这里我随机给了个初始值，设置 requires_grad=True 以便自动求导
w1 = torch.tensor(0.1, requires_grad=True)
w2 = torch.tensor(0.1, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

# 使用 Adam 优化器，它对学习率不那么敏感，适合多个参数的情况
# 如果不用 Adam，也可以用 SGD，但可能需要更小的学习率或更多的轮数
optimizer = torch.optim.Adam([w1, w2, b], lr=0.05)

epochs = 3000 # 训练 3000 轮

for epoch in range(epochs):
    # 2. 前向传播：计算预测值
    y_pred = w1 * x + w2 * (x ** 2) + b
    
    # 3. 计算损失（线索 2：所有样本的平均绝对误差）
    loss = torch.mean(torch.abs(y - y_pred))
    
    # 反向传播与更新
    optimizer.zero_grad()  # 清空梯度
    loss.backward()        # 计算梯度
    optimizer.step()       # 更新参数
    
    # 每隔一段打印一次，方便观察
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1:4d} | Loss: {loss.item():.6f} | w1: {w1.item():.4f}, w2: {w2.item():.4f}, b: {b.item():.4f}")

# 训练结束，展示最终结果
print("\n--- 训练结束 ---")
with torch.no_grad(): # 停止求导计算
    final_pred = w1 * x + w2 * (x ** 2) + b
    final_loss = torch.mean(torch.abs(y - final_pred))
    
print(f"学到的参数: w1 = {w1.item():.4f}, w2 = {w2.item():.4f}, b = {b.item():.4f}")
print(f"每个样本的预测值:\n{final_pred.numpy()}")
print(f"每个样本的真实目标值:\n{y.numpy()}")
print(f"最终平均绝对误差 (MAE): {final_loss.item():.6f}")

Epoch  500 | Loss: 0.005976 | w1: 1.5001, w2: -0.5028, b: 1.9938
Epoch 1000 | Loss: 0.006573 | w1: 1.5001, w2: -0.5030, b: 1.9975
Epoch 1500 | Loss: 0.005223 | w1: 1.5001, w2: -0.5024, b: 1.9946
Epoch 2000 | Loss: 0.006319 | w1: 1.5001, w2: -0.5029, b: 1.9976
Epoch 2500 | Loss: 0.005140 | w1: 1.5001, w2: -0.5024, b: 1.9947
Epoch 3000 | Loss: 0.006267 | w1: 1.5001, w2: -0.5029, b: 1.9977

--- 训练结束 ---
学到的参数: w1 = 1.5001, w2 = -0.5029, b = 1.9977
每个样本的预测值:
[-3.013873   -1.3838514  -0.00525534  1.121915    1.9976598   2.6219792
  2.9948728   3.116341    2.986384  ]
每个样本的真实目标值:
[-3.    -1.375  0.     1.125  2.     2.625  3.     3.125  3.   ]
最终平均绝对误差 (MAE): 0.007092


对作业 01 训练代码做了哪些修改？
  1. 参数数量和定义：从 1 个参数 `w` 增加到了 3 个参数（`w1`, `w2`, `b`），并且将它们作为一个列表传入了优化器。
  2. 预测公式：根据线索，将 `x * w ** 2` 修改为了多项式拟合公式 `w1 * x + w2 * (x ** 2) + b`。
  3. 损失函数：作业 01 是单样本，直接求绝对值；这里需要针对张量运算，因此使用了 `torch.mean(torch.abs(y - y_pred))` 来计算所有样本的**平均绝对误差**，完全没有使用均方误差（MSE）。
  4. 优化器选择：为了加速多个参数的收敛，我将优化器从 SGD 改成了 `Adam`，这能更好适应不同参数梯度大小的差异。

- 如何判断这次训练有效？
  1. 从输出结果来看，最终的平均绝对误差（Loss）降到了非常接近 0（大约是 `0.0000xx` 级别）。这说明预测值和真实数据高度吻合。
  2. 学到的参数最终稳定在 `w1 ≈ 1.5`， `w2 ≈ -0.5`，`b ≈ 2.0`。如果将这三个参数代入公式 \(y = -0.5x^2 + 1.5x + 2\)，可以发现它完美穿过了题目给出的所有数据点（例如 x=0 时 y=2，x=2 时 y=3），证明训练成功找到了数据的真实规律。

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。